# 09 · Augmentation and error analysis

**Level:** Advanced · **Time:** 40–55 minutes.
**Prerequisites:** notebooks 02, 06, and 08; notebook 07 is optional.

**You will:** transform images and boxes together, clip/filter boxes after a
crop, measure a simple detector under a synthetic domain shift, and turn a
failure into a testable improvement hypothesis.

**Demo story:** “The training demo works on a plain background. Does it still
work when the appearance changes?” We will measure actual outputs rather than
assume that an augmentation always improves a model.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image, ImageEnhance
from course_utils import make_scene

image, boxes, labels = make_scene(seed=19, size=256, min_objects=3, max_objects=3)
show_boxes(image, boxes, labels, title="Original training-style scene")
plt.show()


## 1. A geometric transform must also transform the labels

For a horizontal flip of an image of width W:
`new_x1 = W - old_x2` and `new_x2 = W - old_x1`. The y coordinates stay fixed.
We use continuous box boundaries, so this formula uses W rather than W−1.

Photometric changes (brightness/color) normally leave box coordinates unchanged.
Geometric changes (flip, crop, resize, rotation) need corresponding box updates.
A rotated object's axis-aligned box may need to grow; this lesson starts with
flips and crops whose coordinate rules are easy to inspect.

**Presenter prompt:** Ask the audience to predict the flipped position of the
leftmost object before drawing the transformed boxes.


In [ ]:
def flip_horizontal(image, boxes, labels):
    transformed = np.asarray(boxes, dtype=np.float32).reshape(-1, 4).copy()
    transformed[:, [0, 2]] = image.width - transformed[:, [2, 0]]
    return image.transpose(Image.Transpose.FLIP_LEFT_RIGHT), transformed, np.asarray(labels).copy()

flipped, flipped_boxes, flipped_labels = flip_horizontal(image, boxes, labels)
restored_image, restored_boxes, _ = flip_horizontal(flipped, flipped_boxes, flipped_labels)
assert np.allclose(restored_boxes, boxes)
assert np.array_equal(np.asarray(restored_image), np.asarray(image))
brightened = ImageEnhance.Brightness(image).enhance(0.60)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
show_boxes(image, boxes, labels, ax=axes[0], title="Original")
show_boxes(flipped, flipped_boxes, flipped_labels, ax=axes[1], title="Flip image + boxes")
show_boxes(brightened, boxes, labels, ax=axes[2], title="Brightness: boxes unchanged")
plt.tight_layout()
plt.show()


## 2. Crop, translate, clip, then filter

Cropping changes the coordinate origin. Subtract the crop's left/top values,
clip coordinates to its width/height, and discard boxes with no useful visible
region. We use **retained bounding-box area / original box area** as a simple
visibility proxy; it is not the exact visible area of the object.

Filter the corresponding class labels with the same mask. A crop can correctly
produce no objects: preserve boxes with shape `(0, 4)` and labels with shape `(0,)`.


In [ ]:
def crop_with_boxes(image, boxes, labels, crop, min_visibility=0.30, min_side=3):
    left, top, right, bottom = map(int, crop)
    if not (0 <= left < right <= image.width and 0 <= top < bottom <= image.height):
        raise ValueError("Crop must be a nonempty rectangle inside the image.")
    original = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    shifted = original - np.array([left, top, left, top], dtype=np.float32)
    shifted[:, [0, 2]] = np.clip(shifted[:, [0, 2]], 0, right - left)
    shifted[:, [1, 3]] = np.clip(shifted[:, [1, 3]], 0, bottom - top)
    size = np.maximum(shifted[:, 2:] - shifted[:, :2], 0)
    original_area = np.prod(np.maximum(original[:, 2:] - original[:, :2], 0), axis=1)
    visibility = np.prod(size, axis=1) / np.maximum(original_area, 1e-12)
    keep = (size[:, 0] >= min_side) & (size[:, 1] >= min_side) & (visibility >= min_visibility)
    return image.crop((left, top, right, bottom)), shifted[keep], np.asarray(labels)[keep], visibility

crop = (60, 30, 220, 210)
cropped, cropped_boxes, cropped_labels, visibility = crop_with_boxes(image, boxes, labels, crop)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
show_boxes(image, boxes, labels, ax=axes[0], title="Original and requested crop")
from matplotlib.patches import Rectangle
axes[0].add_patch(Rectangle((crop[0], crop[1]), crop[2]-crop[0], crop[3]-crop[1],
                           fill=False, edgecolor="magenta", linewidth=3, linestyle="--"))
show_boxes(cropped, cropped_boxes, cropped_labels, ax=axes[1], title=f"Kept {len(cropped_boxes)}/{len(boxes)} boxes")
plt.tight_layout()
plt.show()
print("Retained box-area fractions:", np.round(visibility, 3))
assert cropped_boxes.shape[1:] == (4,)
if len(cropped_boxes):
    assert np.all(cropped_boxes[:, 2:] > cropped_boxes[:, :2])
    assert np.all(cropped_boxes >= 0)
    assert np.all(cropped_boxes[:, [0, 2]] <= cropped.width)
    assert np.all(cropped_boxes[:, [1, 3]] <= cropped.height)
# A crop containing no object is a valid training example.
_, empty_boxes, empty_labels, _ = crop_with_boxes(
    Image.new("RGB", (20, 20)), np.array([[1, 1, 5, 5]]), np.array([0]), (10, 10, 20, 20))
assert empty_boxes.shape == (0, 4) and empty_labels.shape == (0,)


## 3. Build an intentionally simple appearance baseline

This baseline identifies pixels far from a **source-domain background color**
and surrounds connected components with boxes. It classifies each component
using its occupied area divided by its box area: approximately 1 for a rectangle,
π/4 for a circle, and 1/2 for a triangle. The cutoffs 0.90 and 0.64 are simple
fixed heuristics based on those ideal shapes, not learned model parameters.

This is an actual runnable detector with a deliberately narrow assumption.
It makes shortcuts visible: a good score on matching synthetic data can conceal
dependence on the background. Shape colors in this dataset are randomized
independently of class, so class-color prototypes would be an inappropriate
classifier. We estimate source background from seed 100 and evaluate distinct
seeds 4000–4019, outside the custom training/validation/test seed ranges.
No evaluation labels are used to set the thresholds.


In [ ]:
# Top-left corner is background in the generated scenes (inspect the generator).
source_image, _, _ = make_scene(seed=100)
source_background = np.asarray(source_image, dtype=np.float32)[0, 0]
print("Fixed source-domain background RGB:", source_background)

def color_component_detector(image, minimum_area=60):
    pixels = np.asarray(image, dtype=np.float32)
    foreground = np.linalg.norm(pixels - source_background, axis=2) > 50
    result_boxes, result_labels = [], []
    count, _, stats, _ = cv2.connectedComponentsWithStats(foreground.astype(np.uint8), connectivity=8)
    for x, y, w, h, area in stats[1:]:
        if area >= minimum_area:
            occupancy = area / (w * h)
            class_id = 0 if occupancy > .90 else (1 if occupancy > .64 else 2)
            result_boxes.append([x, y, x+w, y+h])
            result_labels.append(class_id)
    return np.asarray(result_boxes, dtype=np.float32).reshape(-1, 4), np.asarray(result_labels, dtype=np.int64)

baseline_boxes, baseline_labels = color_component_detector(image)
show_boxes(image, baseline_boxes, baseline_labels, title="Measured foreground/component baseline predictions")
plt.show()


## 4. Quantify a domain shift

`shift=True` asks the generator for changed visual conditions. Inspect the
examples and its implementation in `course_utils.py` to understand what changed.
Both groups use the same seed list, preserving geometry and class labels.
Background brightness, object brightness, and image noise change together.
This is a reproducible stress test, not a causal proof that one of those
properties alone explains a score difference.

The component baseline has no meaningful confidence scores. We report one
fixed operating point: class-aware, one-to-one precision/recall at IoU 0.50.
Matching is greedy in component order. Do not label this result mAP.


In [ ]:
def count_matches(pred_boxes, pred_labels, gt_boxes, gt_labels, iou_threshold=.5):
    unmatched = set(range(len(gt_boxes)))
    tp = 0
    for box, class_id in zip(pred_boxes, pred_labels):
        candidates = [j for j in sorted(unmatched) if gt_labels[j] == class_id]
        if not candidates:
            continue
        candidates_boxes = np.asarray(gt_boxes)[candidates]
        intersection = np.prod(np.maximum(np.minimum(box[2:], candidates_boxes[:, 2:]) -
                                            np.maximum(box[:2], candidates_boxes[:, :2]), 0), axis=1)
        area = np.prod(np.maximum(box[2:] - box[:2], 0))
        gt_area = np.prod(np.maximum(candidates_boxes[:, 2:] - candidates_boxes[:, :2], 0), axis=1)
        ious = intersection / np.maximum(area + gt_area - intersection, 1e-12)
        best = int(np.argmax(ious))
        if ious[best] >= iou_threshold:
            unmatched.remove(candidates[best])
            tp += 1
    return tp, len(pred_boxes) - tp, len(unmatched)

shift_rows = []
for shifted in [False, True]:
    totals = np.zeros(3, dtype=int)
    for seed in range(4000, 4020):
        sample, gt_boxes, gt_labels = make_scene(seed=seed, shift=shifted)
        pred_boxes, pred_labels = color_component_detector(sample)
        totals += count_matches(pred_boxes, pred_labels, gt_boxes, gt_labels)
    tp, fp, fn = map(int, totals)
    shift_rows.append({"domain": "shifted" if shifted else "source", "TP": tp, "FP": fp, "FN": fn,
                       "precision": tp / max(tp+fp, 1), "recall": tp / max(tp+fn, 1)})
shift_results = pd.DataFrame(shift_rows)
display(shift_results.round(3))
shift_results.plot.bar(x="domain", y=["precision", "recall"], ylim=(0, 1.05), rot=0,
                       title="Measured component baseline · IoU 0.50", figsize=(7, 4))
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, shifted in zip(axes, [False, True]):
    sample, _, _ = make_scene(seed=4001, shift=shifted)
    pred_boxes, pred_labels = color_component_detector(sample)
    show_boxes(sample, pred_boxes, pred_labels, ax=ax,
               title="Shifted predictions" if shifted else "Source predictions")
plt.tight_layout()
plt.show()


## 5. Optional: inspect your trained YOLO model on both domains

These are real predictions from the saved custom checkpoint. The plot is a
small qualitative check, not an aggregate robustness score. A short training
run may perform poorly even before the shift. Keep that baseline in view.


In [ ]:
manifest_path = ROOT / "artifacts" / "custom_model.json"
if os.environ.get("COURSE_OFFLINE", "0") == "1" or not manifest_path.exists():
    print("Optional checkpoint inference skipped. Train notebook 07 to enable it online.")
else:
    from ultralytics import YOLO
    from course_utils import choose_device
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    weights = Path(manifest["best_weights"])
    if weights.exists():
        custom_model = YOLO(str(weights))
        scenes = [make_scene(seed=4001, shift=shifted)[0] for shifted in [False, True]]
        results = custom_model.predict(scenes, imgsz=256, conf=.10, device=choose_device(), verbose=False)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, result, name in zip(axes, results, ["Source", "Shifted"]):
            ax.imshow(result.plot()[..., ::-1]); ax.axis("off")
            ax.set_title(f"{name}: {len(result.boxes)} model predictions")
        plt.tight_layout(); plt.show()
    else:
        print("Saved weights moved or missing:", weights)


## 6. Turn observations into an error-analysis plan

| Error | Evidence to inspect | One controlled experiment |
|---|---|---|
| Missed object | Ground truth with no matching prediction | Collect more examples of that size/appearance |
| Wrong class | Good overlap, wrong label | Check annotation consistency and class confusion |
| Poor localization | Correct class, low IoU | Inspect box tightness and image resolution |
| Duplicate | Multiple predictions for one object | Inspect NMS settings and crowded scenes |
| Background false alarm | Prediction with no object | Add representative negative images |
| Domain shift | Held-out subgroup performance changes | Add realistic variation to training, then remeasure |

**Live exercise:** Change `min_visibility` from 0.30 to 0.80 and rerun the crop.
Explain every removed label. Then choose one realistic change in your intended
application and propose an augmentation. Do not augment validation/test images
randomly during normal evaluation; keep evaluation conditions reproducible.

**Advanced exercise:** Replace fixed-source-background subtraction with a
per-image background estimate. Keep the evaluation seeds fixed and measure both
groups again. Separate fitting data from evaluation data. Which shapes still
fail because noise, occlusion, or low contrast changes their component geometry?

<details><summary>Solutions / discussion</summary>

Raising minimum visibility can only keep the same number or fewer boxes. An
object may remain partly visible after its label is removed; aggressive cropping
can therefore teach unintended negatives. Color jitter is reasonable when color
varies in the real task, but harmful if color defines the label. A per-image
background estimate can help the segmentation step without repairing dependence
on the foreground mask. Measure the actual outcome; neither fix guarantees improvement.
</details>

**Checkpoint:** Demonstrate a correctly transformed box and state a measured
failure with one proposed experiment. Next: tracking/video and model internals.

**Primary references:** [TorchVision transforms and bounding boxes](https://docs.pytorch.org/vision/stable/transforms.html),
[Ultralytics augmentation guide](https://docs.ultralytics.com/guides/yolo-data-augmentation/).
